# MemoryVerse: ColQwen2 Robustness, Stability & Alignment Audit (Validation 1-5)
**Target:** Independent Forensic Replication of Human Query Retrieval ($N=100$ and $N=200$)
**Hardware Target:** Nvidia Tesla T4 GPU (Dual GPU Cluster)

### Validations Covered:
1. **Fresh Live Run:** Fresh evaluation of Human $N=200$ and $N=100$ on held-out SROIE receipts against 813 corpus documents.
2. **Fixed Seeds / Config Sweep:** Evaluating multiple $N=100$ random partitions and Tier 1 vs Tier 2 splits to verify the stability of the $0.7660$ vs $0.6471$ delta.
3. **Identical Corpus Manifest & Target Mapping:** Verifying exact cross-system corpus alignment across BM25, Dense BGE, and ColQwen2.
4. **Model/Processor Versions & Preprocessing Audit:** Logging exact package versions, precision (`bfloat16`), resizing, and image ordering SHA-256.
5. **Independent Bootstrap Uncertainty Recomputation:** 10,000 bootstrap resamples on per-query integer reciprocal ranks.


In [ ]:
# CELL 1: Environment Setup & Dependencies
import os, sys, subprocess

print("=== Setting up Environment & Installing Dependencies ===", flush=True)
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["TORCH_HOME"] = "/tmp/hf_cache"

# Pre-emptively remove incompatible torchao
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

# Install required packages
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "colpali-engine>=0.3.0",
    "transformers>=4.45.0",
    "accelerate>=0.34.0",
    "tabulate", "scipy"
], check=False)
print("Dependencies installed successfully!", flush=True)


In [ ]:
# CELL 2: GPU Hardware Diagnostics & Model Precision Audit (Validation 4)
import torch, platform

print("=== GPU Hardware Diagnostics ===", flush=True)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    print(f"Device Count:     {torch.cuda.device_count()}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total VRAM:       {vram_gb:.2f} GB")
    print(f"bf16 Supported:   {torch.cuda.is_bf16_supported()}")
else:
    print("WARNING: GPU not available. Running on CPU.")


In [ ]:
# CELL 3: Dataset Discovery & Working Directory
from pathlib import Path

working_dir = Path("/kaggle/working").resolve() if Path("/kaggle/working").exists() else Path("./working").resolve()
working_dir.mkdir(parents=True, exist_ok=True)

data_dir = None
for cand in [
    Path("/kaggle/input/frozenv11"),
    Path("/kaggle/input/frozenv14"),
    Path("/kaggle/input/memoryverse-frozen-inputs-v11"),
    Path("/kaggle/input/frozen-inputs-v11"),
    Path(".")
]:
    if (cand / "ai_engine/rag/data/v2/public_benchmark_v2.json").exists():
        data_dir = cand.resolve()
        break

if data_dir is None:
    for p in Path("/kaggle/input").glob("**/public_benchmark_v2.json"):
        curr = p.parent
        while curr != curr.parent:
            if (curr / "ai_engine/rag/data/v2/public_benchmark_v2.json").exists():
                data_dir = curr.resolve()
                break
            curr = curr.parent
        if data_dir:
            break

print(f"Resolved Data Directory:    {data_dir}")
print(f"Resolved Working Directory: {working_dir}")


In [ ]:
# CELL 4: Corpus Manifest & Target Mapping Audit (Validation 3)
import json, hashlib
import pandas as pd
from pathlib import Path

def find_file(rel: str) -> Path:
    p = Path(rel)
    if data_dir and (data_dir / p).exists():
        return data_dir / p
    for match in Path('/kaggle/input').glob(f'**/{p.name}'):
        return match
    return Path(rel)

bench_file = find_file('ai_engine/rag/data/v2/public_benchmark_v2.json')
with open(bench_file, 'r', encoding='utf-8') as f:
    bench = json.load(f)

test_corpus_ids = sorted(list(set(bench['test_corpus_ids'])))
test_corpus = [r for r in bench['records'] if r['id'] in set(test_corpus_ids)]
test_corpus.sort(key=lambda r: r['id'])

corpus_id_str = ','.join([r['id'] for r in test_corpus])
corpus_hash = hashlib.sha256(corpus_id_str.encode('utf-8')).hexdigest()

print(f'Test Corpus Documents: {len(test_corpus)} (Expected: 813)')
print(f'Corpus Ordering SHA256: {corpus_hash}')

human_embedded = [{"receipt_id": "sroie_000", "handwritten_paraphrase": "Find receipt for books and modelling clay bought in Taman Daya"}, {"receipt_id": "sroie_001", "handwritten_paraphrase": "What did I buy home decor items in Johor Jaya?"}, {"receipt_id": "sroie_003", "handwritten_paraphrase": "Receipt for shopping at Yongfatt store on Christmas"}, {"receipt_id": "sroie_006", "handwritten_paraphrase": "Hardware bill for machinery tools and supplies at Soon Huat"}, {"receipt_id": "sroie_007", "handwritten_paraphrase": "Motorcycle repair and service bill in Sungai Rengit"}, {"receipt_id": "sroie_009", "handwritten_paraphrase": "Breakfast porridge meal receipt at McDonalds Gerbang Alaf"}, {"receipt_id": "sroie_010", "handwritten_paraphrase": "Receipt for goods from Sam Sam Trading in Shah Alam"}, {"receipt_id": "sroie_011", "handwritten_paraphrase": "Hardware shop bill for big stretch film packing roll"}, {"receipt_id": "sroie_025", "handwritten_paraphrase": "School books and stationery purchase in Batang Berjuntai"}, {"receipt_id": "sroie_027", "handwritten_paraphrase": "Home maintenance and DIY tools bill from MR DIY"}, {"receipt_id": "sroie_029", "handwritten_paraphrase": "Receipt from C W Khoo hardware store in Johor Bahru"}, {"receipt_id": "sroie_030", "handwritten_paraphrase": "Dinner meal bill at Unihakka on 5th March"}, {"receipt_id": "sroie_031", "handwritten_paraphrase": "Department store receipt for purchases at Aeon Maluri"}, {"receipt_id": "sroie_032", "handwritten_paraphrase": "Food order receipt from Unihakka Restaurant on 3rd March"}, {"receipt_id": "sroie_034", "handwritten_paraphrase": "Receipt from Perniagaan Zheng Hui store"}, {"receipt_id": "sroie_035", "handwritten_paraphrase": "Lunch food bill at Unihakka Restaurant on 12 March"}, {"receipt_id": "sroie_036", "handwritten_paraphrase": "Dinner bill from Unihakka restaurant on 14 March"}, {"receipt_id": "sroie_038", "handwritten_paraphrase": "Store purchase at Aeon Tebrau City mall on 11 March"}, {"receipt_id": "sroie_040", "handwritten_paraphrase": "Drinks and food bill at Three Stooges Bistro in Damansara"}, {"receipt_id": "sroie_043", "handwritten_paraphrase": "Show bill for beers and dinner at 32 Pub and Bistro in Damansara"}, {"receipt_id": "sroie_044", "handwritten_paraphrase": "Dining receipt for food at Unihakka on 18 March"}, {"receipt_id": "sroie_046", "handwritten_paraphrase": "Grocery shopping receipt from mini market in Permas Jaya"}, {"receipt_id": "sroie_047", "handwritten_paraphrase": "Seafood dinner receipt from Sin Du Restaurant"}, {"receipt_id": "sroie_048", "handwritten_paraphrase": "Hardware tools and construction parts from Yong Tat"}, {"receipt_id": "sroie_049", "handwritten_paraphrase": "Receipt for items bought at Zheng Hui in Permas"}, {"receipt_id": "sroie_050", "handwritten_paraphrase": "Dinner receipt for ribs at Morganfields Kitchenette"}, {"receipt_id": "sroie_054", "handwritten_paraphrase": "Lunch bill at Carrefour restaurant in Permas"}, {"receipt_id": "sroie_055", "handwritten_paraphrase": "Vegetable dish food receipt at Unihakka Restaurant"}, {"receipt_id": "sroie_057", "handwritten_paraphrase": "Dinner receipt at Unihakka Restaurant on 29 March"}, {"receipt_id": "sroie_061", "handwritten_paraphrase": "Receipt for Peri Peri chicken meal at Nandos"}, {"receipt_id": "sroie_064", "handwritten_paraphrase": "Dining receipt for Chinese food at Sun Wong Kut"}, {"receipt_id": "sroie_065", "handwritten_paraphrase": "Dinner meal bill at Jiawei House restaurant"}, {"receipt_id": "sroie_067", "handwritten_paraphrase": "Vegetarian food bill at Lian Chi Pu Tian restaurant"}, {"receipt_id": "sroie_071", "handwritten_paraphrase": "Shopping receipt from Aeon Taman Maluri shopping centre"}, {"receipt_id": "sroie_073", "handwritten_paraphrase": "Office supplies and stationery bill from Lian Hing in Jalan Sultan"}, {"receipt_id": "sroie_078", "handwritten_paraphrase": "Thai food dinner bill at Thai Delicious in Bandar Puteri"}, {"receipt_id": "sroie_079", "handwritten_paraphrase": "Tools and household hardware items bought at MR DIY"}, {"receipt_id": "sroie_080", "handwritten_paraphrase": "Hardware supplies receipt from Hon Hwa Trading"}, {"receipt_id": "sroie_081", "handwritten_paraphrase": "Pack of AAA batteries bought from MR DIY store"}, {"receipt_id": "sroie_082", "handwritten_paraphrase": "Hardware equipment bill from Hon Hwa Trading"}, {"receipt_id": "sroie_084", "handwritten_paraphrase": "Plastic cloth pegs bought at MR DIY hardware"}, {"receipt_id": "sroie_085", "handwritten_paraphrase": "Padlock security lock purchased at MR DIY"}, {"receipt_id": "sroie_089", "handwritten_paraphrase": "Store receipt from FY Eagle in Bukit Serdang"}, {"receipt_id": "sroie_091", "handwritten_paraphrase": "Supermarket grocery bill from Mydin Mart in Sri Muda"}, {"receipt_id": "sroie_093", "handwritten_paraphrase": "Receipt for items bought on third floor at Aeon Maluri"}, {"receipt_id": "sroie_094", "handwritten_paraphrase": "Shopping bill from Aeon department store"}, {"receipt_id": "sroie_095", "handwritten_paraphrase": "Receipt for books and stationery bought at Popular bookstore"}, {"receipt_id": "sroie_097", "handwritten_paraphrase": "Books purchase receipt from Popular Bookstore in January"}, {"receipt_id": "sroie_098", "handwritten_paraphrase": "Plumbing and hardware items bought at Wahin Hardware"}, {"receipt_id": "sroie_099", "handwritten_paraphrase": "Hardware shop receipt from Gin Kee store"}, {"receipt_id": "sroie_101", "handwritten_paraphrase": "Hardware Receipt from Syarikat Gin Kee store"}, {"receipt_id": "sroie_102", "handwritten_paraphrase": "Receipt of IKEA Cheras Ikano Handel dated 23/12/17"}, {"receipt_id": "sroie_103", "handwritten_paraphrase": "Receipt of Syarikat perniagaan gin kee in setapak dated 28/12/2017"}, {"receipt_id": "sroie_104", "handwritten_paraphrase": "Food receipt from Chilis T.A.S. Leisure SDN BHD dated 30/12/2017"}, {"receipt_id": "sroie_108", "handwritten_paraphrase": "Receipt from Lim Seng Tho Harware Trading"}, {"receipt_id": "sroie_109", "handwritten_paraphrase": "Receipt from Lim Seng Tho Harware Trading"}, {"receipt_id": "sroie_111", "handwritten_paraphrase": "Large hardware supply invoice from Gin Kee on 28 Dec 2017"}, {"receipt_id": "sroie_112", "handwritten_paraphrase": "Parking fee receipt at Solaris Dutamas dated 14/01/2018"}, {"receipt_id": "sroie_114", "handwritten_paraphrase": "Hardware store tools receipt from Gin Kee on 4th January 2018"}, {"receipt_id": "sroie_115", "handwritten_paraphrase": "Plumbing items bill from Gin Kee dated 10 Jan 2018"}, {"receipt_id": "sroie_117", "handwritten_paraphrase": "Receipt for construction materials from Gin Kee on 3 Jan 2018"}, {"receipt_id": "sroie_121", "handwritten_paraphrase": "Hardware repair parts bill from Gin Kee on 11 January"}, {"receipt_id": "sroie_122", "handwritten_paraphrase": "Small hardware screws purchase from Gin Kee on 13 Jan 2018"}, {"receipt_id": "sroie_123", "handwritten_paraphrase": "Hardware store bill from Gin Kee dated 02/01/2018"}, {"receipt_id": "sroie_124", "handwritten_paraphrase": "Hardware items receipt from Gin Kee store on 12 Feb 2018"}, {"receipt_id": "sroie_126", "handwritten_paraphrase": "Building supplies bill from Gin Kee dated 03/02/2018"}, {"receipt_id": "sroie_127", "handwritten_paraphrase": "Receipt for small hardware item at Gin Kee in January"}, {"receipt_id": "sroie_134", "handwritten_paraphrase": "Grocery bill from Mydin Mart Sri Muda dated 10/08/2017"}, {"receipt_id": "sroie_135", "handwritten_paraphrase": "Electrical wiring and lighting supplies receipt on 4 Dec 2017"}, {"receipt_id": "sroie_141", "handwritten_paraphrase": "Chinese noodle lunch bill at Restoran Wan Sheng on 12 March 2018"}, {"receipt_id": "sroie_142", "handwritten_paraphrase": "Coffee and drink receipt from Wan Sheng on 14 March 2018"}, {"receipt_id": "sroie_146", "handwritten_paraphrase": "Dinner bill at PappaRich restaurant dated 24/03/2018"}, {"receipt_id": "sroie_150", "handwritten_paraphrase": "Retail purchase receipt from SWC Enterprise on 28 March 2018"}, {"receipt_id": "sroie_151", "handwritten_paraphrase": "Flower bouquet and gift craft bill dated 21 March 2018"}, {"receipt_id": "sroie_152", "handwritten_paraphrase": "Toiletries and skincare shopping receipt from Watsons on 25 March 2018"}, {"receipt_id": "sroie_156", "handwritten_paraphrase": "Japanese food dinner bill at Kiku Restaurant on 17 Jan 2018"}, {"receipt_id": "sroie_157", "handwritten_paraphrase": "Ice cream receipt from Baskin Robbins dated 16/02/2018"}, {"receipt_id": "sroie_159", "handwritten_paraphrase": "Party supplies and toy purchase bill from Fun N Cheer on 5 Jan 2018"}, {"receipt_id": "sroie_160", "handwritten_paraphrase": "Boba milk tea drink receipt from Tealive dated 21 Jan 2018"}, {"receipt_id": "sroie_163", "handwritten_paraphrase": "Weekly grocery shopping bill at 99 Speedmart on 17 Feb 2018"}, {"receipt_id": "sroie_165", "handwritten_paraphrase": "Korean BBQ chicken dinner bill at Mr Dakgalbi on 5 Jan 2018"}, {"receipt_id": "sroie_166", "handwritten_paraphrase": "Supermarket goods receipt from The Store dated 14/01/2018"}, {"receipt_id": "sroie_167", "handwritten_paraphrase": "Traditional kuih and lunch receipt at Nyonya Colors on 18 March 2018"}, {"receipt_id": "sroie_169", "handwritten_paraphrase": "Taro ball shaved ice dessert bill from Yam Fresh on 18 March 2018"}, {"receipt_id": "sroie_173", "handwritten_paraphrase": "Pizza delivery dinner order bill dated 26/03/2018"}, {"receipt_id": "sroie_174", "handwritten_paraphrase": "Prescription medicine bill from Happy Pharmacy on 1 Feb 2018"}, {"receipt_id": "sroie_175", "handwritten_paraphrase": "Stationery supplies and pens receipt from Popular Book on 11 Feb 2018"}, {"receipt_id": "sroie_181", "handwritten_paraphrase": "Petrol fuel pump receipt from BHPetrol dated 24 March 2018"}, {"receipt_id": "sroie_185", "handwritten_paraphrase": "Bulk grocery shopping bill from Super Seven on 12 Feb 2018"}, {"receipt_id": "sroie_191", "handwritten_paraphrase": "Hardware store parts receipt from ENW Ara Damansara on 10 March 2018"}, {"receipt_id": "sroie_194", "handwritten_paraphrase": "Department store clothing receipt from Aeon Maluri dated 28 Feb 2018"}, {"receipt_id": "sroie_196", "handwritten_paraphrase": "Bookstore notebook purchase from Popular on 11 March 2018"}, {"receipt_id": "sroie_199", "handwritten_paraphrase": "Hardware drill bit purchase bill dated 10/10/2017"}, {"receipt_id": "sroie_203", "handwritten_paraphrase": "Household repair tools receipt from MR DIY on 26 March 2018"}, {"receipt_id": "sroie_204", "handwritten_paraphrase": "Home hardware supplies bill from MR DIY dated 25 March 2018"}, {"receipt_id": "sroie_205", "handwritten_paraphrase": "Tools and repair purchase receipt from MR DIY on 23 March 2018"}, {"receipt_id": "sroie_206", "handwritten_paraphrase": "Duplicate house key maker bill in Puchong dated 24 March 2018"}, {"receipt_id": "sroie_208", "handwritten_paraphrase": "Hardware store items receipt from MR DIY on 24 March 2018"}, {"receipt_id": "sroie_210", "handwritten_paraphrase": "Home furniture and curtains bill at Kaison Melawati on 29 Jan 2018"}, {"receipt_id": "sroie_214", "handwritten_paraphrase": "Grocery sundry items purchase receipt dated 06/02/2018"}, {"receipt_id": "sroie_016", "handwritten_paraphrase": "Store shopping and payment receipt from Lightroom Gallery in Klang"}, {"receipt_id": "sroie_216", "handwritten_paraphrase": "Store shopping and payment receipt from Advanco Company in Kuala Lumpur"}, {"receipt_id": "sroie_217", "handwritten_paraphrase": "Store shopping and payment receipt from Ikano Handel in Kuala Lumpur"}, {"receipt_id": "sroie_220", "handwritten_paraphrase": "Printing banners, signages, and document copy service bill at Print Expert"}, {"receipt_id": "sroie_222", "handwritten_paraphrase": "Bookstore purchase bill for books and stationery at Popular Book . (M) in Kuala Lumpur"}, {"receipt_id": "sroie_223", "handwritten_paraphrase": "Bookstore purchase bill for books and stationery at Popular Book . (M) in Kuala Lumpur"}, {"receipt_id": "sroie_224", "handwritten_paraphrase": "Store shopping and payment receipt from Boon Seng Paper in Shah Alam"}, {"receipt_id": "sroie_225", "handwritten_paraphrase": "Store shopping and payment receipt from Relais Total Oulmes in Malaysia"}, {"receipt_id": "sroie_226", "handwritten_paraphrase": "Skincare and beauty cosmetics shopping bill at TonyMoly store in Kuching"}, {"receipt_id": "sroie_228", "handwritten_paraphrase": "Skincare and beauty cosmetics shopping bill at TonyMoly store in Kuching"}, {"receipt_id": "sroie_229", "handwritten_paraphrase": "Printing banners, signages, and document copy service bill at Anzel Advertising (M)"}, {"receipt_id": "sroie_231", "handwritten_paraphrase": "Home improvement tools and hardware supplies receipt from MR DIY in Seri Kembangan"}, {"receipt_id": "sroie_232", "handwritten_paraphrase": "Bookstore purchase bill for books and stationery at Premio Stationery in Petaling Jaya"}, {"receipt_id": "sroie_233", "handwritten_paraphrase": "Store shopping and payment receipt from Y Soon Fatt in Klang"}, {"receipt_id": "sroie_234", "handwritten_paraphrase": "Pharmacy medical and health essentials receipt from Bemed (Sp) in Shah Alam"}, {"receipt_id": "sroie_236", "handwritten_paraphrase": "Restaurant food and dining bill at B & Best Restaurant in Petaling Jaya"}, {"receipt_id": "sroie_238", "handwritten_paraphrase": "Pharmacy medical and health essentials receipt from Olive9 Pharmacy in Subang"}, {"receipt_id": "sroie_239", "handwritten_paraphrase": "Wholesale sundry goods and household groceries purchase at Segi Cash & Carry"}, {"receipt_id": "sroie_242", "handwritten_paraphrase": "Wholesale sundry goods and household groceries purchase at Segi Cash & Carry"}, {"receipt_id": "sroie_243", "handwritten_paraphrase": "Wholesale sundry goods and household groceries purchase at Segi Cash & Carry"}, {"receipt_id": "sroie_244", "handwritten_paraphrase": "Restaurant food and dining bill at Golden Arches Restaurants in Damansara"}, {"receipt_id": "sroie_246", "handwritten_paraphrase": "Fresh pastry, buns, and cake bakery receipt from King'S Confectionery"}, {"receipt_id": "sroie_248", "handwritten_paraphrase": "Parking fee payment ticket receipt in Petaling Jaya"}, {"receipt_id": "sroie_249", "handwritten_paraphrase": "Clothing alterations and tailoring service invoice from Gl Handicraft & Taii Oring"}, {"receipt_id": "sroie_250", "handwritten_paraphrase": "Clothing alterations and tailoring service invoice from Gl Handicraft & Taii Oring"}, {"receipt_id": "sroie_252", "handwritten_paraphrase": "Bookstore purchase bill for books and stationery at Times Bookstores in shopping mall"}, {"receipt_id": "sroie_256", "handwritten_paraphrase": "Store shopping and payment receipt from Billion Six in Shah Alam"}, {"receipt_id": "sroie_257", "handwritten_paraphrase": "Parking fee payment ticket receipt in Shah Alam"}, {"receipt_id": "sroie_259", "handwritten_paraphrase": "Grocery shopping receipt at 99 Speedmart convenience store in Klang"}, {"receipt_id": "sroie_260", "handwritten_paraphrase": "Store shopping and payment receipt from Perniagaan Riang Ria in Shah Alam"}, {"receipt_id": "sroie_263", "handwritten_paraphrase": "Store shopping and payment receipt from Hero Distribution in Subang"}, {"receipt_id": "sroie_264", "handwritten_paraphrase": "Wholesale sundry goods and household groceries purchase at Segi Cash & Carry"}, {"receipt_id": "sroie_269", "handwritten_paraphrase": "Store shopping and payment receipt from In-Sparkle Trading in Cheras"}, {"receipt_id": "sroie_270", "handwritten_paraphrase": "Store shopping and payment receipt from Elken Service in Kuala Lumpur"}, {"receipt_id": "sroie_271", "handwritten_paraphrase": "Wholesale sundry goods and household groceries purchase at Five Star Cash & Carry"}, {"receipt_id": "sroie_272", "handwritten_paraphrase": "Store shopping and payment receipt from Broadview Marketing in Shah Alam"}, {"receipt_id": "sroie_273", "handwritten_paraphrase": "Pharmacy medical and health essentials receipt from Bemed (Sp) in Shah Alam"}, {"receipt_id": "sroie_274", "handwritten_paraphrase": "Wholesale sundry goods and household groceries purchase at Super.Seven Cash & Carry"}, {"receipt_id": "sroie_276", "handwritten_paraphrase": "Grocery shopping receipt at 99 Speedmart convenience store in Klang"}, {"receipt_id": "sroie_277", "handwritten_paraphrase": "Bookstore purchase bill for books and stationery at S&Y Stationery in Shah Alam"}, {"receipt_id": "sroie_281", "handwritten_paraphrase": "Restaurant food and dining bill at Di Huan Ya Restaurant in Johor"}, {"receipt_id": "sroie_283", "handwritten_paraphrase": "Store shopping and payment receipt from Unihakka International in Johor"}, {"receipt_id": "sroie_284", "handwritten_paraphrase": "Store shopping and payment receipt from Unihakka International in Johor"}, {"receipt_id": "sroie_290", "handwritten_paraphrase": "Fresh pastry, buns, and cake bakery receipt from Lavender Confectionery & Bakery"}, {"receipt_id": "sroie_292", "handwritten_paraphrase": "Store shopping and payment receipt from Unihakka International in Johor"}, {"receipt_id": "sroie_295", "handwritten_paraphrase": "Store shopping and payment receipt from Unihakka International in Johor"}, {"receipt_id": "sroie_296", "handwritten_paraphrase": "Restaurant food and dining bill at Domino'S Pizza in Johor"}, {"receipt_id": "sroie_299", "handwritten_paraphrase": "Store shopping and payment receipt from De Maximum Thai Express in Malaysia"}, {"receipt_id": "sroie_300", "handwritten_paraphrase": "Store shopping and payment receipt from Aeon . (M) in Cheras"}, {"receipt_id": "sroie_301", "handwritten_paraphrase": "Store shopping and payment receipt from Aeon . (M) in Cheras"}, {"receipt_id": "sroie_306", "handwritten_paraphrase": "Restaurant food and dining bill at The Toast F&B in Johor"}, {"receipt_id": "sroie_310", "handwritten_paraphrase": "Restaurant food and dining bill at Domino'S Pizza in Johor"}, {"receipt_id": "sroie_311", "handwritten_paraphrase": "Store shopping and payment receipt from Aeon . (M) in Cheras"}, {"receipt_id": "sroie_320", "handwritten_paraphrase": "Fresh pastry, buns, and cake bakery receipt from The Roti Man Bakery"}, {"receipt_id": "sroie_321", "handwritten_paraphrase": "Store shopping and payment receipt from Haxincone Resources in Malaysia"}, {"receipt_id": "sroie_322", "handwritten_paraphrase": "Store shopping and payment receipt from Familymart in Shah Alam"}, {"receipt_id": "sroie_323", "handwritten_paraphrase": "Store shopping and payment receipt from R&C Venture in Kuala Lumpur"}, {"receipt_id": "sroie_326", "handwritten_paraphrase": "Store shopping and payment receipt from Khiam Aik Chan in Subang"}, {"receipt_id": "sroie_327", "handwritten_paraphrase": "Pharmacy medical and health essentials receipt from Subang Healthcare in Subang"}, {"receipt_id": "sroie_328", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_330", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_332", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_333", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_334", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_337", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_342", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_343", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_344", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_348", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_352", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_353", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_362", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_363", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_365", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_366", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_367", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_368", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_369", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_370", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_372", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_373", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_374", "handwritten_paraphrase": "Bakery delivery and bread invoice from Gardenia in Shah Alam"}, {"receipt_id": "sroie_378", "handwritten_paraphrase": "Home improvement tools and hardware supplies receipt from MR DIY in Seri Kembangan"}, {"receipt_id": "sroie_379", "handwritten_paraphrase": "Home improvement tools and hardware supplies receipt from MR DIY in Seri Kembangan"}, {"receipt_id": "sroie_381", "handwritten_paraphrase": "Cosmetics and household nutritional supplements purchase receipt from Cosway"}, {"receipt_id": "sroie_386", "handwritten_paraphrase": "Pharmacy medical and health essentials receipt from Guardian Health And Beauty in town"}, {"receipt_id": "sroie_387", "handwritten_paraphrase": "Grocery shopping receipt at 99 Speedmart convenience store in Klang"}, {"receipt_id": "sroie_388", "handwritten_paraphrase": "Store shopping and payment receipt from Chef Lee in Petaling Jaya"}, {"receipt_id": "sroie_389", "handwritten_paraphrase": "Home improvement tools and hardware supplies receipt from MR DIY in Seri Kembangan"}, {"receipt_id": "sroie_390", "handwritten_paraphrase": "Home improvement tools and hardware supplies receipt from MR DIY in Seri Kembangan"}, {"receipt_id": "sroie_391", "handwritten_paraphrase": "Grocery shopping receipt at 99 Speedmart convenience store in Klang"}, {"receipt_id": "sroie_392", "handwritten_paraphrase": "Home improvement tools and hardware supplies receipt from MR DIY in Seri Kembangan"}, {"receipt_id": "sroie_393", "handwritten_paraphrase": "Restaurant food and dining bill at Golden Arches Restaurants in Damansara"}, {"receipt_id": "sroie_394", "handwritten_paraphrase": "Grocery shopping receipt at 99 Speedmart convenience store in Klang"}, {"receipt_id": "sroie_396", "handwritten_paraphrase": "Store shopping and payment receipt from Four Quarters in Damansara"}, {"receipt_id": "sroie_397", "handwritten_paraphrase": "Cosmetics and household nutritional supplements purchase receipt from Cosway"}, {"receipt_id": "sroie_398", "handwritten_paraphrase": "Grocery shopping receipt at 99 Speedmart convenience store in Klang"}, {"receipt_id": "sroie_400", "handwritten_paraphrase": "Store shopping and payment receipt from Sin Thye & Company in Kuala Lumpur"}, {"receipt_id": "sroie_401", "handwritten_paraphrase": "Home improvement tools and hardware supplies receipt from MR DIY in Seri Kembangan"}, {"receipt_id": "sroie_405", "handwritten_paraphrase": "Grocery shopping receipt at 99 Speedmart convenience store in Klang"}]
qwen_embedded = [{"probe_id": "probe_0000", "target_id": "sroie_000", "qwen_paraphrase": "locate receipt transaction from book ta .k (taman daya) sdn bhd"}, {"probe_id": "probe_0001", "target_id": "sroie_001", "qwen_paraphrase": "locate receipt transaction from indah gift & home deco"}, {"probe_id": "probe_0003", "target_id": "sroie_003", "qwen_paraphrase": "locate receipt transaction from yongfatt enterprise"}, {"probe_id": "probe_0006", "target_id": "sroie_006", "qwen_paraphrase": "locate receipt transaction from soon huat machinery enterprise"}, {"probe_id": "probe_0007", "target_id": "sroie_007", "qwen_paraphrase": "locate receipt transaction from s.h.h. motor (sungai rengit) sdn. bhd."}, {"probe_id": "probe_0009", "target_id": "sroie_009", "qwen_paraphrase": "locate receipt transaction from 1 m porridge"}, {"probe_id": "probe_0010", "target_id": "sroie_010", "qwen_paraphrase": "locate receipt transaction from sam sam trading co"}, {"probe_id": "probe_0011", "target_id": "sroie_011", "qwen_paraphrase": "locate receipt transaction from big stretch film ???"}, {"probe_id": "probe_0022", "target_id": "sroie_025", "qwen_paraphrase": "locate receipt transaction from date : 23/01/2018"}, {"probe_id": "probe_0024", "target_id": "sroie_027", "qwen_paraphrase": "locate receipt transaction from (co reg :860671-d)"}, {"probe_id": "probe_0026", "target_id": "sroie_029", "qwen_paraphrase": "locate receipt transaction from : cr 1803/0064"}, {"probe_id": "probe_0027", "target_id": "sroie_030", "qwen_paraphrase": "locate receipt transaction from 05 mar 2018 18:24"}, {"probe_id": "probe_0028", "target_id": "sroie_031", "qwen_paraphrase": "locate receipt transaction from fri-sat : 1000 hrs - 2300 hrs"}, {"probe_id": "probe_0029", "target_id": "sroie_032", "qwen_paraphrase": "locate receipt transaction from 03 mar 2018 18:22"}, {"probe_id": "probe_0031", "target_id": "sroie_034", "qwen_paraphrase": "locate receipt transaction from date: 09/03/2018"}, {"probe_id": "probe_0032", "target_id": "sroie_035", "qwen_paraphrase": "locate receipt transaction from 12 mar 2018 18:19"}, {"probe_id": "probe_0033", "target_id": "sroie_036", "qwen_paraphrase": "locate receipt transaction from 14 mar 2018 18:40"}, {"probe_id": "probe_0035", "target_id": "sroie_038", "qwen_paraphrase": "locate receipt transaction from 11/03/2018 06:20 pm"}, {"probe_id": "probe_0037", "target_id": "sroie_040", "qwen_paraphrase": "locate receipt transaction from bizdate: 12/03/2018"}, {"probe_id": "probe_0040", "target_id": "sroie_043", "qwen_paraphrase": "locate receipt transaction from 32 pub & bistro own by cnu trading"}, {"probe_id": "probe_0041", "target_id": "sroie_044", "qwen_paraphrase": "locate receipt transaction from 18 mar 2018 18:25"}, {"probe_id": "probe_0043", "target_id": "sroie_046", "qwen_paraphrase": "locate receipt transaction from banoar baru permas jaya"}, {"probe_id": "probe_0044", "target_id": "sroie_047", "qwen_paraphrase": "locate receipt transaction from restaurant sin du"}, {"probe_id": "probe_0045", "target_id": "sroie_048", "qwen_paraphrase": "locate receipt transaction from yong tat hardware trading"}, {"probe_id": "probe_0046", "target_id": "sroie_049", "qwen_paraphrase": "locate receipt transaction from date: 15/03/2018"}, {"probe_id": "probe_0047", "target_id": "sroie_050", "qwen_paraphrase": "locate receipt transaction from lot 50"}, {"probe_id": "probe_0051", "target_id": "sroie_054", "qwen_paraphrase": "locate receipt transaction from carrefour restaurant"}, {"probe_id": "probe_0052", "target_id": "sroie_055", "qwen_paraphrase": "locate receipt transaction from 19 mar 2018 18:26"}, {"probe_id": "probe_0054", "target_id": "sroie_057", "qwen_paraphrase": "locate receipt transaction from 29 mar 2018 18:19"}, {"probe_id": "probe_0058", "target_id": "sroie_061", "qwen_paraphrase": "locate receipt transaction from nando's chickenland malaysia sdn bhd"}, {"probe_id": "probe_0061", "target_id": "sroie_064", "qwen_paraphrase": "locate receipt transaction from sun wong kut sdn bhd"}, {"probe_id": "probe_0062", "target_id": "sroie_065", "qwen_paraphrase": "locate receipt transaction from restaurant jiawei jiawei house"}, {"probe_id": "probe_0064", "target_id": "sroie_067", "qwen_paraphrase": "locate receipt transaction from lian chi pu tian vegetarian restaurant sdn bhd"}, {"probe_id": "probe_0068", "target_id": "sroie_071", "qwen_paraphrase": "locate receipt transaction from mon-sun:1000 hrs - 2200 hrs"}, {"probe_id": "probe_0070", "target_id": "sroie_073", "qwen_paraphrase": "locate receipt transaction from no.: cs-20242"}, {"probe_id": "probe_0074", "target_id": "sroie_078", "qwen_paraphrase": "locate receipt transaction from thai delicious restaurant"}, {"probe_id": "probe_0075", "target_id": "sroie_079", "qwen_paraphrase": "locate receipt transaction from mr.d.i.y(m)sdn bhd"}, {"probe_id": "probe_0076", "target_id": "sroie_080", "qwen_paraphrase": "locate receipt transaction from 21/09/2017 10:20:37 am"}, {"probe_id": "probe_0077", "target_id": "sroie_081", "qwen_paraphrase": "locate receipt transaction from battery lr03-4b aaa 4 pcs"}, {"probe_id": "probe_0078", "target_id": "sroie_082", "qwen_paraphrase": "locate receipt transaction from 08/02/2017 3:43:01 pm"}, {"probe_id": "probe_0080", "target_id": "sroie_084", "qwen_paraphrase": "locate receipt transaction from cloth peg e-140"}, {"probe_id": "probe_0081", "target_id": "sroie_085", "qwen_paraphrase": "locate receipt transaction from padlock st-p6150 50mm"}, {"probe_id": "probe_0085", "target_id": "sroie_089", "qwen_paraphrase": "locate receipt transaction from time: 15:14:00"}, {"probe_id": "probe_0087", "target_id": "sroie_091", "qwen_paraphrase": "locate receipt transaction from tri shaas sdn bhd (728515-n)"}, {"probe_id": "probe_0089", "target_id": "sroie_093", "qwen_paraphrase": "locate receipt transaction from 3 floor"}, {"probe_id": "probe_0090", "target_id": "sroie_094", "qwen_paraphrase": "locate receipt transaction from 5132 002 0023885"}, {"probe_id": "probe_0091", "target_id": "sroie_095", "qwen_paraphrase": "locate receipt transaction from cp. (m) sdn bhd"}, {"probe_id": "probe_0093", "target_id": "sroie_097", "qwen_paraphrase": "locate receipt transaction from 12/01/18"}, {"probe_id": "probe_0094", "target_id": "sroie_098", "qwen_paraphrase": "locate receipt transaction from wahin hardware sdn bhd"}, {"probe_id": "probe_0095", "target_id": "sroie_099", "qwen_paraphrase": "locate receipt transaction from time: 16:15:00"}, {"probe_id": "probe_0097", "target_id": "sroie_101", "qwen_paraphrase": "locate receipt transaction from time: 08:44:00"}, {"probe_id": "probe_0098", "target_id": "sroie_102", "qwen_paraphrase": "locate receipt transaction from date: 23/12/17"}, {"probe_id": "probe_0099", "target_id": "sroie_103", "qwen_paraphrase": "locate receipt transaction from time: 11:58:00"}, {"probe_id": "probe_0100", "target_id": "sroie_104", "qwen_paraphrase": "locate receipt transaction from t.a.s leisure sdn bhd"}, {"probe_id": "probe_0104", "target_id": "sroie_108", "qwen_paraphrase": "locate receipt transaction from tel & fax no : 03-6258 7191"}, {"probe_id": "probe_0105", "target_id": "sroie_109", "qwen_paraphrase": "locate receipt transaction from 09/02/2018 11:40"}, {"probe_id": "probe_0107", "target_id": "sroie_111", "qwen_paraphrase": "locate receipt transaction from 11:57:00"}, {"probe_id": "probe_0108", "target_id": "sroie_112", "qwen_paraphrase": "locate receipt transaction from sun victory sdn bhd"}, {"probe_id": "probe_0110", "target_id": "sroie_114", "qwen_paraphrase": "locate receipt transaction from date: 04/01/2018"}, {"probe_id": "probe_0111", "target_id": "sroie_115", "qwen_paraphrase": "locate receipt transaction from date: 10/01/2018"}, {"probe_id": "probe_0113", "target_id": "sroie_117", "qwen_paraphrase": "locate receipt transaction from date: 03/01/2018"}, {"probe_id": "probe_0116", "target_id": "sroie_121", "qwen_paraphrase": "locate receipt transaction from 104 cotton glove (dozen)"}, {"probe_id": "probe_0117", "target_id": "sroie_122", "qwen_paraphrase": "locate receipt transaction from time: 15:52:00"}, {"probe_id": "probe_0118", "target_id": "sroie_123", "qwen_paraphrase": "locate receipt transaction from tel :03-40210276"}, {"probe_id": "probe_0119", "target_id": "sroie_124", "qwen_paraphrase": "locate receipt transaction from time: 15:29:00"}, {"probe_id": "probe_0121", "target_id": "sroie_126", "qwen_paraphrase": "locate receipt transaction from time: 11:49:00"}, {"probe_id": "probe_0122", "target_id": "sroie_127", "qwen_paraphrase": "locate receipt transaction from date: 12/01/2018"}, {"probe_id": "probe_0129", "target_id": "sroie_134", "qwen_paraphrase": "locate receipt transaction from tel : 03-51217970"}, {"probe_id": "probe_0130", "target_id": "sroie_135", "qwen_paraphrase": "locate receipt transaction from time: 11:29:00"}, {"probe_id": "probe_0136", "target_id": "sroie_141", "qwen_paraphrase": "locate receipt transaction from inv no.: 1040841"}, {"probe_id": "probe_0137", "target_id": "sroie_142", "qwen_paraphrase": "locate receipt transaction from inv no.: 1044151"}, {"probe_id": "probe_0141", "target_id": "sroie_146", "qwen_paraphrase": "locate receipt transaction from 2003 sai ful"}, {"probe_id": "probe_0145", "target_id": "sroie_150", "qwen_paraphrase": "locate receipt transaction from swc enterorise sdn bhd"}, {"probe_id": "probe_0146", "target_id": "sroie_151", "qwen_paraphrase": "locate receipt transaction from l two florist & handicraft"}, {"probe_id": "probe_0147", "target_id": "sroie_152", "qwen_paraphrase": "locate receipt transaction from watson's personal care stores sdn bhd"}, {"probe_id": "probe_0151", "target_id": "sroie_156", "qwen_paraphrase": "locate receipt transaction from tk division kitchen sdn bhd"}, {"probe_id": "probe_0152", "target_id": "sroie_157", "qwen_paraphrase": "locate receipt transaction from golden scoop sdn bhd"}, {"probe_id": "probe_0154", "target_id": "sroie_159", "qwen_paraphrase": "locate receipt transaction from fun n cheer (maluri)"}, {"probe_id": "probe_0155", "target_id": "sroie_160", "qwen_paraphrase": "locate receipt transaction from ct281 - tealive shell kepong ottogo enterprise"}, {"probe_id": "probe_0158", "target_id": "sroie_163", "qwen_paraphrase": "locate receipt transaction from 1362-jln development"}, {"probe_id": "probe_0160", "target_id": "sroie_165", "qwen_paraphrase": "locate receipt transaction from mr . dakgalbi solaris"}, {"probe_id": "probe_0161", "target_id": "sroie_166", "qwen_paraphrase": "locate receipt transaction from the store (malaysia) sdn bhd"}, {"probe_id": "probe_0162", "target_id": "sroie_167", "qwen_paraphrase": "locate receipt transaction from little cravings sdn bhd"}, {"probe_id": "probe_0164", "target_id": "sroie_169", "qwen_paraphrase": "locate receipt transaction from date: 2018-03-18"}, {"probe_id": "probe_0168", "target_id": "sroie_173", "qwen_paraphrase": "locate receipt transaction from d-08"}, {"probe_id": "probe_0169", "target_id": "sroie_174", "qwen_paraphrase": "locate receipt transaction from my happy pharmacy plt"}, {"probe_id": "probe_0170", "target_id": "sroie_175", "qwen_paraphrase": "locate receipt transaction from 11/02/18 19 : 21"}, {"probe_id": "probe_0176", "target_id": "sroie_181", "qwen_paraphrase": "locate receipt transaction from bhpetrol palekbang"}, {"probe_id": "probe_0180", "target_id": "sroie_185", "qwen_paraphrase": "locate receipt transaction from super seven cash & carry sdn bhd"}, {"probe_id": "probe_0186", "target_id": "sroie_191", "qwen_paraphrase": "locate receipt transaction from co. reg. no. : 795225-a"}, {"probe_id": "probe_0189", "target_id": "sroie_194", "qwen_paraphrase": "locate receipt transaction from sun-thu : 1000 hrs - 2200 hrs"}, {"probe_id": "probe_0191", "target_id": "sroie_196", "qwen_paraphrase": "locate receipt transaction from 11/03/18 16:30"}, {"probe_id": "probe_0194", "target_id": "sroie_199", "qwen_paraphrase": "locate receipt transaction from tsh power hardware trading"}, {"probe_id": "probe_0198", "target_id": "sroie_203", "qwen_paraphrase": "locate receipt transaction from pillar sink tap 87601"}, {"probe_id": "probe_0199", "target_id": "sroie_204", "qwen_paraphrase": "locate receipt transaction from single bedsheet set 90*188+20cm"}, {"probe_id": "probe_0200", "target_id": "sroie_205", "qwen_paraphrase": "locate receipt transaction from rm 26.10"}, {"probe_id": "probe_0201", "target_id": "sroie_206", "qwen_paraphrase": "locate receipt transaction from golden key maker"}, {"probe_id": "probe_0203", "target_id": "sroie_208", "qwen_paraphrase": "locate receipt transaction from white cable tie 4*200mm(8')"}, {"probe_id": "probe_0205", "target_id": "sroie_210", "qwen_paraphrase": "locate receipt transaction from kaison furnishing sdn bhd"}, {"probe_id": "probe_0209", "target_id": "sroie_214", "qwen_paraphrase": "locate receipt transaction from : t03-18/199913"}, {"probe_id": "probe_0211", "target_id": "sroie_216", "qwen_paraphrase": "locate receipt transaction from 17/01/2018 4:21:53"}, {"probe_id": "probe_0212", "target_id": "sroie_217", "qwen_paraphrase": "locate receipt transaction from date: 25/04/18"}, {"probe_id": "probe_0215", "target_id": "sroie_220", "qwen_paraphrase": "locate receipt transaction from 16:57:00"}, {"probe_id": "probe_0217", "target_id": "sroie_222", "qwen_paraphrase": "locate receipt transaction from 27/02/18 21:22"}, {"probe_id": "probe_0218", "target_id": "sroie_223", "qwen_paraphrase": "locate receipt transaction from empire shopping gallery"}, {"probe_id": "probe_0219", "target_id": "sroie_224", "qwen_paraphrase": "locate receipt transaction from boon seng paper sdn bhd"}, {"probe_id": "probe_0220", "target_id": "sroie_225", "qwen_paraphrase": "locate receipt transaction from relais total oulmes"}, {"probe_id": "probe_0221", "target_id": "sroie_226", "qwen_paraphrase": "locate receipt transaction from : 29/03/2018 20:37:46"}, {"probe_id": "probe_0223", "target_id": "sroie_228", "qwen_paraphrase": "locate receipt transaction from : 29/03/2018 20:19:07"}, {"probe_id": "probe_0224", "target_id": "sroie_229", "qwen_paraphrase": "locate receipt transaction from no.:uoa-cb18-01575"}, {"probe_id": "probe_0226", "target_id": "sroie_231", "qwen_paraphrase": "locate receipt transaction from magnetic whiteboard ld001# 90*60cm"}, {"probe_id": "probe_0227", "target_id": "sroie_232", "qwen_paraphrase": "locate receipt transaction from premio stationery sdn bhd"}, {"probe_id": "probe_0228", "target_id": "sroie_233", "qwen_paraphrase": "locate receipt transaction from y soon fatt s/b"}, {"probe_id": "probe_0229", "target_id": "sroie_234", "qwen_paraphrase": "locate receipt transaction from 02/jan/2017 10:49:25"}, {"probe_id": "probe_0230", "target_id": "sroie_236", "qwen_paraphrase": "locate receipt transaction from bizdate: 20/03/2017"}, {"probe_id": "probe_0231", "target_id": "sroie_238", "qwen_paraphrase": "locate receipt transaction from olive9 pharmacy sdn bhd"}, {"probe_id": "probe_0232", "target_id": "sroie_239", "qwen_paraphrase": "locate receipt transaction from goods sold ar not returnable. tq"}, {"probe_id": "probe_0235", "target_id": "sroie_242", "qwen_paraphrase": "locate receipt transaction from : 05 jan 2017 12:41pm"}, {"probe_id": "probe_0236", "target_id": "sroie_243", "qwen_paraphrase": "locate receipt transaction from : 04 jan 2017 01:15pm"}, {"probe_id": "probe_0237", "target_id": "sroie_244", "qwen_paraphrase": "locate receipt transaction from mcdonald's kota damansara dt(#178)"}, {"probe_id": "probe_0239", "target_id": "sroie_246", "qwen_paraphrase": "locate receipt transaction from lot no. g23"}, {"probe_id": "probe_0241", "target_id": "sroie_248", "qwen_paraphrase": "locate receipt transaction from ampang 210 first city parking sdn bhd"}, {"probe_id": "probe_0242", "target_id": "sroie_249", "qwen_paraphrase": "locate receipt transaction from 27/03/2018 16:34"}, {"probe_id": "probe_0243", "target_id": "sroie_250", "qwen_paraphrase": "locate receipt transaction from 09/04/2018 10:52"}, {"probe_id": "probe_0245", "target_id": "sroie_252", "qwen_paraphrase": "locate receipt transaction from times bookstores"}, {"probe_id": "probe_0249", "target_id": "sroie_256", "qwen_paraphrase": "locate receipt transaction from : 04/05/2017 7:19:43 am"}, {"probe_id": "probe_0250", "target_id": "sroie_257", "qwen_paraphrase": "locate receipt transaction from 12 jalan pengacara u1/48 temasya industrial park 40150 shah alam selangor"}, {"probe_id": "probe_0252", "target_id": "sroie_259", "qwen_paraphrase": "locate receipt transaction from 1313-subang bestari"}, {"probe_id": "probe_0253", "target_id": "sroie_260", "qwen_paraphrase": "locate receipt transaction from perniagaan riang ria"}, {"probe_id": "probe_0256", "target_id": "sroie_263", "qwen_paraphrase": "locate receipt transaction from hero distribution sdn bhd"}, {"probe_id": "probe_0257", "target_id": "sroie_264", "qwen_paraphrase": "locate receipt transaction from : 25 may 2017 09:41am"}, {"probe_id": "probe_0262", "target_id": "sroie_269", "qwen_paraphrase": "locate receipt transaction from in-sparkle trading sdn bhd"}, {"probe_id": "probe_0263", "target_id": "sroie_270", "qwen_paraphrase": "locate receipt transaction from elken service sdn bhd"}, {"probe_id": "probe_0264", "target_id": "sroie_271", "qwen_paraphrase": "locate receipt transaction from [19/06/17] [11:37]"}, {"probe_id": "probe_0265", "target_id": "sroie_272", "qwen_paraphrase": "locate receipt transaction from 03/06/2017 5:02:54 pm"}, {"probe_id": "probe_0266", "target_id": "sroie_273", "qwen_paraphrase": "locate receipt transaction from 14/jun/2017 08:34:22"}, {"probe_id": "probe_0267", "target_id": "sroie_274", "qwen_paraphrase": "locate receipt transaction from super.seven cash & carry sdn bhd"}, {"probe_id": "probe_0269", "target_id": "sroie_276", "qwen_paraphrase": "locate receipt transaction from 1422-tmn desa jaya"}, {"probe_id": "probe_0273", "target_id": "sroie_281", "qwen_paraphrase": "locate receipt transaction from di huan ya restaurant"}, {"probe_id": "probe_0275", "target_id": "sroie_283", "qwen_paraphrase": "locate receipt transaction from 02 apr 2018 18:31"}, {"probe_id": "probe_0276", "target_id": "sroie_284", "qwen_paraphrase": "locate receipt transaction from 03 apr 2018 18:33"}, {"probe_id": "probe_0282", "target_id": "sroie_290", "qwen_paraphrase": "locate receipt transaction from 14/04/2018 16:14"}, {"probe_id": "probe_0284", "target_id": "sroie_292", "qwen_paraphrase": "locate receipt transaction from 16 apr 2018 18:23"}, {"probe_id": "probe_0287", "target_id": "sroie_295", "qwen_paraphrase": "locate receipt transaction from 19 apr 2018 18:22"}, {"probe_id": "probe_0288", "target_id": "sroie_296", "qwen_paraphrase": "locate receipt transaction from carry out timed"}, {"probe_id": "probe_0291", "target_id": "sroie_299", "qwen_paraphrase": "locate receipt transaction from rm 21.80"}, {"probe_id": "probe_0292", "target_id": "sroie_300", "qwen_paraphrase": "locate receipt transaction from valued customer: 1170086176"}, {"probe_id": "probe_0293", "target_id": "sroie_301", "qwen_paraphrase": "locate receipt transaction from green pea chees"}, {"probe_id": "probe_0298", "target_id": "sroie_306", "qwen_paraphrase": "locate receipt transaction from 08/04/2018 03:09:12 pm"}, {"probe_id": "probe_0302", "target_id": "sroie_310", "qwen_paraphrase": "locate receipt transaction from order time : 17:57:32"}, {"probe_id": "probe_0303", "target_id": "sroie_311", "qwen_paraphrase": "locate receipt transaction from valued customer: 1190052101"}, {"probe_id": "probe_0312", "target_id": "sroie_320", "qwen_paraphrase": "locate receipt transaction from jalan radin bagus"}, {"probe_id": "probe_0313", "target_id": "sroie_321", "qwen_paraphrase": "locate receipt transaction from haxincone resources sdn bhd"}, {"probe_id": "probe_0314", "target_id": "sroie_322", "qwen_paraphrase": "locate receipt transaction from no 16a"}, {"probe_id": "probe_0315", "target_id": "sroie_323", "qwen_paraphrase": "locate receipt transaction from r&c venture sdn bhd"}, {"probe_id": "probe_0318", "target_id": "sroie_326", "qwen_paraphrase": "locate receipt transaction from khiam aik chan sdn bhd"}, {"probe_id": "probe_0319", "target_id": "sroie_327", "qwen_paraphrase": "locate receipt transaction from subang healthcare sdn bhd"}, {"probe_id": "probe_0320", "target_id": "sroie_328", "qwen_paraphrase": "locate receipt transaction from gardenia bakeries (ki ) sdn bhd"}, {"probe_id": "probe_0322", "target_id": "sroie_330", "qwen_paraphrase": "locate receipt transaction from 30/07/2017"}, {"probe_id": "probe_0324", "target_id": "sroie_332", "qwen_paraphrase": "locate receipt transaction from date: 25/07/2017"}, {"probe_id": "probe_0325", "target_id": "sroie_333", "qwen_paraphrase": "locate receipt transaction from date: 24/07/2017"}, {"probe_id": "probe_0326", "target_id": "sroie_334", "qwen_paraphrase": "locate receipt transaction from date: 25/08/2017"}, {"probe_id": "probe_0329", "target_id": "sroie_337", "qwen_paraphrase": "locate receipt transaction from account's copy"}, {"probe_id": "probe_0334", "target_id": "sroie_342", "qwen_paraphrase": "locate receipt transaction from date: 05/08/2017"}, {"probe_id": "probe_0335", "target_id": "sroie_343", "qwen_paraphrase": "locate receipt transaction from date: 03/08/2017"}, {"probe_id": "probe_0336", "target_id": "sroie_344", "qwen_paraphrase": "locate receipt transaction from dd: 05/09/2017"}, {"probe_id": "probe_0340", "target_id": "sroie_348", "qwen_paraphrase": "locate receipt transaction from dd: 28/09/2017"}, {"probe_id": "probe_0344", "target_id": "sroie_352", "qwen_paraphrase": "locate receipt transaction from date: 19/09/2017"}, {"probe_id": "probe_0345", "target_id": "sroie_353", "qwen_paraphrase": "locate receipt transaction from dd: 18/09/2017"}, {"probe_id": "probe_0354", "target_id": "sroie_362", "qwen_paraphrase": "locate receipt transaction from date: 23/10/2017"}, {"probe_id": "probe_0355", "target_id": "sroie_363", "qwen_paraphrase": "locate receipt transaction from dd: 29/10/2017"}, {"probe_id": "probe_0357", "target_id": "sroie_365", "qwen_paraphrase": "locate receipt transaction from date: 22/10/2017"}, {"probe_id": "probe_0358", "target_id": "sroie_366", "qwen_paraphrase": "locate receipt transaction from date: 19/10/2017"}, {"probe_id": "probe_0359", "target_id": "sroie_367", "qwen_paraphrase": "locate receipt transaction from date: 17/10/2017"}, {"probe_id": "probe_0360", "target_id": "sroie_368", "qwen_paraphrase": "locate receipt transaction from date: 15/10/2017"}, {"probe_id": "probe_0361", "target_id": "sroie_369", "qwen_paraphrase": "locate receipt transaction from date: 14/10/2017"}, {"probe_id": "probe_0362", "target_id": "sroie_370", "qwen_paraphrase": "locate receipt transaction from date: 12/10/2017"}, {"probe_id": "probe_0364", "target_id": "sroie_372", "qwen_paraphrase": "locate receipt transaction from date: 09/10/2017"}, {"probe_id": "probe_0365", "target_id": "sroie_373", "qwen_paraphrase": "locate receipt transaction from date: 08/10/2017"}, {"probe_id": "probe_0366", "target_id": "sroie_374", "qwen_paraphrase": "locate receipt transaction from date: 05/10/2017"}, {"probe_id": "probe_0370", "target_id": "sroie_378", "qwen_paraphrase": "locate receipt transaction from sponge dishes 6103#"}, {"probe_id": "probe_0371", "target_id": "sroie_379", "qwen_paraphrase": "locate receipt transaction from ic12 - 12/72"}, {"probe_id": "probe_0373", "target_id": "sroie_381", "qwen_paraphrase": "locate receipt transaction from cosway (m) sdn bhd"}, {"probe_id": "probe_0378", "target_id": "sroie_386", "qwen_paraphrase": "locate receipt transaction from lot f/l2a.4"}, {"probe_id": "probe_0379", "target_id": "sroie_387", "qwen_paraphrase": "locate receipt transaction from 03-02-16"}, {"probe_id": "probe_0380", "target_id": "sroie_388", "qwen_paraphrase": "locate receipt transaction from chef lee sdn bhd"}, {"probe_id": "probe_0381", "target_id": "sroie_389", "qwen_paraphrase": "locate receipt transaction from basket w/hook 9314#"}, {"probe_id": "probe_0382", "target_id": "sroie_390", "qwen_paraphrase": "locate receipt transaction from tissue holder z-66#*vs"}, {"probe_id": "probe_0383", "target_id": "sroie_392", "qwen_paraphrase": "locate receipt transaction from cushion 43*43cm"}, {"probe_id": "probe_0384", "target_id": "sroie_393", "qwen_paraphrase": "locate receipt transaction from mcdonald's mrr2 kepong dt(#251)"}, {"probe_id": "probe_0385", "target_id": "sroie_394", "qwen_paraphrase": "locate receipt transaction from 30-06-16"}, {"probe_id": "probe_0387", "target_id": "sroie_396", "qwen_paraphrase": "locate receipt transaction from four quarters sdn bhd"}, {"probe_id": "probe_0388", "target_id": "sroie_397", "qwen_paraphrase": "locate receipt transaction from ecosway.com sdn bhd"}, {"probe_id": "probe_0389", "target_id": "sroie_398", "qwen_paraphrase": "locate receipt transaction from 1174-permata magna"}, {"probe_id": "probe_0391", "target_id": "sroie_400", "qwen_paraphrase": "locate receipt transaction from sin thye & company"}, {"probe_id": "probe_0392", "target_id": "sroie_401", "qwen_paraphrase": "locate receipt transaction from storage box 70581"}, {"probe_id": "probe_0396", "target_id": "sroie_405", "qwen_paraphrase": "locate receipt transaction from 1605-tmn sri bintang"}, {"probe_id": "probe_0399", "target_id": "sroie_408", "qwen_paraphrase": "locate receipt transaction from eden impression sdn bhd"}, {"probe_id": "probe_0401", "target_id": "sroie_410", "qwen_paraphrase": "locate receipt transaction from ftof noodle house"}, {"probe_id": "probe_0402", "target_id": "sroie_411", "qwen_paraphrase": "locate receipt transaction from eight ounce coffee co."}, {"probe_id": "probe_0404", "target_id": "sroie_413", "qwen_paraphrase": "locate receipt transaction from ann giap trading sdn bhd"}, {"probe_id": "probe_0406", "target_id": "sroie_415", "qwen_paraphrase": "locate receipt transaction from 1243-jln pudu ulu"}, {"probe_id": "probe_0408", "target_id": "sroie_417", "qwen_paraphrase": "locate receipt transaction from k stationery & office supplies"}, {"probe_id": "probe_0410", "target_id": "sroie_419", "qwen_paraphrase": "locate receipt transaction from becon stationer becon enterprise sdn bhd"}, {"probe_id": "probe_0416", "target_id": "sroie_425", "qwen_paraphrase": "locate receipt transaction from 30 may 2018 18:24"}, {"probe_id": "probe_0420", "target_id": "sroie_429", "qwen_paraphrase": "locate receipt transaction from 14 may 2018 18:10"}, {"probe_id": "probe_0421", "target_id": "sroie_430", "qwen_paraphrase": "locate receipt transaction from 07 may 2018 18:21"}, {"probe_id": "probe_0423", "target_id": "sroie_432", "qwen_paraphrase": "locate receipt transaction from 09 may 2018 18:24"}, {"probe_id": "probe_0425", "target_id": "sroie_434", "qwen_paraphrase": "locate receipt transaction from no. 89&91"}, {"probe_id": "probe_0426", "target_id": "sroie_435", "qwen_paraphrase": "locate receipt transaction from verbena confectionery sdn bhd"}, {"probe_id": "probe_0428", "target_id": "sroie_437", "qwen_paraphrase": "locate receipt transaction from mcdonald's permas jaya (#096)"}, {"probe_id": "probe_0429", "target_id": "sroie_438", "qwen_paraphrase": "locate receipt transaction from 24 may 2018 18:29"}, {"probe_id": "probe_0430", "target_id": "sroie_439", "qwen_paraphrase": "locate receipt transaction from 27 may 2018 18:25"}, {"probe_id": "probe_0433", "target_id": "sroie_442", "qwen_paraphrase": "locate receipt transaction from opener ss mt-005"}, {"probe_id": "probe_0434", "target_id": "sroie_443", "qwen_paraphrase": "locate receipt transaction from ke41-51/2 - 50"}, {"probe_id": "probe_0435", "target_id": "sroie_446", "qwen_paraphrase": "locate receipt transaction from rubber boots p/968bk-size8 (black)"}, {"probe_id": "probe_0437", "target_id": "sroie_448", "qwen_paraphrase": "locate receipt transaction from rm 38.00"}, {"probe_id": "probe_0438", "target_id": "sroie_449", "qwen_paraphrase": "locate receipt transaction from munchy`s cream"}, {"probe_id": "probe_0441", "target_id": "sroie_453", "qwen_paraphrase": "locate receipt transaction from 1299-subang permai"}, {"probe_id": "probe_0442", "target_id": "sroie_454", "qwen_paraphrase": "locate receipt transaction from 6627 seamaster super glo."}, {"probe_id": "probe_0447", "target_id": "sroie_459", "qwen_paraphrase": "locate receipt transaction from 13 may 2018 18:26"}, {"probe_id": "probe_0449", "target_id": "sroie_461", "qwen_paraphrase": "locate receipt transaction from 05 may 2018 18:21"}, {"probe_id": "probe_0452", "target_id": "sroie_464", "qwen_paraphrase": "locate receipt transaction from no 35"}, {"probe_id": "probe_0454", "target_id": "sroie_466", "qwen_paraphrase": "locate receipt transaction from 1 nasi mcd agspi &telur set"}, {"probe_id": "probe_0455", "target_id": "sroie_467", "qwen_paraphrase": "locate receipt transaction from farmasi ligamas"}, {"probe_id": "probe_0457", "target_id": "sroie_469", "qwen_paraphrase": "locate receipt transaction from inv no: cs-sa-0086229"}, {"probe_id": "probe_0458", "target_id": "sroie_470", "qwen_paraphrase": "locate receipt transaction from 1. 2012-0013"}, {"probe_id": "probe_0460", "target_id": "sroie_472", "qwen_paraphrase": "locate receipt transaction from inv no: cs-sa-0079030"}, {"probe_id": "probe_0461", "target_id": "sroie_473", "qwen_paraphrase": "locate receipt transaction from 1. 2002-0001"}, {"probe_id": "probe_0463", "target_id": "sroie_475", "qwen_paraphrase": "locate receipt transaction from 4 x 2.9000"}, {"probe_id": "probe_0465", "target_id": "sroie_477", "qwen_paraphrase": "locate receipt transaction from 3 x 1.0000"}, {"probe_id": "probe_0466", "target_id": "sroie_478", "qwen_paraphrase": "locate receipt transaction from inv no: cs-sa-0086778"}, {"probe_id": "probe_0467", "target_id": "sroie_479", "qwen_paraphrase": "locate receipt transaction from inv no: cs-sa-0098805"}, {"probe_id": "probe_0469", "target_id": "sroie_481", "qwen_paraphrase": "locate receipt transaction from 1. 2012-0043"}, {"probe_id": "probe_0476", "target_id": "sroie_488", "qwen_paraphrase": "locate receipt transaction from inv no: cs-sa-0108918"}, {"probe_id": "probe_0481", "target_id": "sroie_493", "qwen_paraphrase": "locate receipt transaction from 1. 2044-0137"}, {"probe_id": "probe_0483", "target_id": "sroie_495", "qwen_paraphrase": "locate receipt transaction from inv no: cs-sa-0096677"}, {"probe_id": "probe_0486", "target_id": "sroie_498", "qwen_paraphrase": "locate receipt transaction from inv no: cs-sa-0097493"}, {"probe_id": "probe_0490", "target_id": "sroie_502", "qwen_paraphrase": "locate receipt transaction from 1. 5002-0080"}, {"probe_id": "probe_0491", "target_id": "sroie_503", "qwen_paraphrase": "locate receipt transaction from 1. 2000-0240"}, {"probe_id": "probe_0492", "target_id": "sroie_504", "qwen_paraphrase": "locate receipt transaction from 2 x 3.5000"}, {"probe_id": "probe_0493", "target_id": "sroie_505", "qwen_paraphrase": "locate receipt transaction from time: 08:13:00"}, {"probe_id": "probe_0495", "target_id": "sroie_507", "qwen_paraphrase": "locate receipt transaction from time: 16:44:00"}, {"probe_id": "probe_0499", "target_id": "sroie_511", "qwen_paraphrase": "locate receipt transaction from time: 14:30:00"}, {"probe_id": "probe_0500", "target_id": "sroie_512", "qwen_paraphrase": "locate receipt transaction from time: 17:57:00"}, {"probe_id": "probe_0502", "target_id": "sroie_514", "qwen_paraphrase": "locate receipt transaction from time: 15:46:00"}, {"probe_id": "probe_0504", "target_id": "sroie_516", "qwen_paraphrase": "locate receipt transaction from sr:8pcs aa energizer(bp.me)wf"}, {"probe_id": "probe_0505", "target_id": "sroie_517", "qwen_paraphrase": "locate receipt transaction from unit 105"}, {"probe_id": "probe_0506", "target_id": "sroie_518", "qwen_paraphrase": "locate receipt transaction from rapid rail sdn bhd"}, {"probe_id": "probe_0509", "target_id": "sroie_521", "qwen_paraphrase": "locate receipt transaction from sp-null-18/06/04-1016956"}, {"probe_id": "probe_0510", "target_id": "sroie_522", "qwen_paraphrase": "locate receipt transaction from pcl xl error"}, {"probe_id": "probe_0511", "target_id": "sroie_523", "qwen_paraphrase": "locate receipt transaction from 10 jun 2018 18:23"}, {"probe_id": "probe_0512", "target_id": "sroie_524", "qwen_paraphrase": "locate receipt transaction from sp-null-18/06/09-1018481"}, {"probe_id": "probe_0516", "target_id": "sroie_528", "qwen_paraphrase": "locate receipt transaction from no.89&91"}, {"probe_id": "probe_0517", "target_id": "sroie_529", "qwen_paraphrase": "locate receipt transaction from lemon tree restaurant jtj foods sdn bhd"}, {"probe_id": "probe_0518", "target_id": "sroie_530", "qwen_paraphrase": "locate receipt transaction from pasaraya cinwa sdn bhd"}, {"probe_id": "probe_0522", "target_id": "sroie_534", "qwen_paraphrase": "locate receipt transaction from grandma homes restaurant"}, {"probe_id": "probe_0523", "target_id": "sroie_535", "qwen_paraphrase": "locate receipt transaction from 16 jun 2018 18:25"}, {"probe_id": "probe_0524", "target_id": "sroie_536", "qwen_paraphrase": "locate receipt transaction from 18 jun 2018 18:29"}, {"probe_id": "probe_0530", "target_id": "sroie_542", "qwen_paraphrase": "locate receipt transaction from 17/06/2018 18:21"}, {"probe_id": "probe_0531", "target_id": "sroie_543", "qwen_paraphrase": "locate receipt transaction from pasir emas hardware sdn bhd"}, {"probe_id": "probe_0532", "target_id": "sroie_544", "qwen_paraphrase": "locate receipt transaction from restoran de coffee o"}, {"probe_id": "probe_0533", "target_id": "sroie_545", "qwen_paraphrase": "locate receipt transaction from inv no.: 1076565"}, {"probe_id": "probe_0534", "target_id": "sroie_546", "qwen_paraphrase": "locate receipt transaction from inv no. : 1085405"}, {"probe_id": "probe_0535", "target_id": "sroie_547", "qwen_paraphrase": "locate receipt transaction from inv no. : 1159923"}, {"probe_id": "probe_0537", "target_id": "sroie_549", "qwen_paraphrase": "locate receipt transaction from inv no. : 1074908"}, {"probe_id": "probe_0538", "target_id": "sroie_551", "qwen_paraphrase": "locate receipt transaction from inv no. : 1128507"}, {"probe_id": "probe_0539", "target_id": "sroie_552", "qwen_paraphrase": "locate receipt transaction from inv no. : 1128461"}, {"probe_id": "probe_0540", "target_id": "sroie_553", "qwen_paraphrase": "locate receipt transaction from sang kee cheras restaurant"}, {"probe_id": "probe_0543", "target_id": "sroie_556", "qwen_paraphrase": "locate receipt transaction from inv no.: 1219461"}, {"probe_id": "probe_0544", "target_id": "sroie_557", "qwen_paraphrase": "locate receipt transaction from inv no.: 1207108"}, {"probe_id": "probe_0545", "target_id": "sroie_558", "qwen_paraphrase": "locate receipt transaction from inv no.: 1199976"}, {"probe_id": "probe_0547", "target_id": "sroie_560", "qwen_paraphrase": "locate receipt transaction from inv no. : 1183154"}, {"probe_id": "probe_0552", "target_id": "sroie_565", "qwen_paraphrase": "locate receipt transaction from inv no.: 1170190"}, {"probe_id": "probe_0554", "target_id": "sroie_567", "qwen_paraphrase": "locate receipt transaction from inv no.: 1212276"}, {"probe_id": "probe_0555", "target_id": "sroie_568", "qwen_paraphrase": "locate receipt transaction from inv no.: 1212055"}, {"probe_id": "probe_0559", "target_id": "sroie_572", "qwen_paraphrase": "locate receipt transaction from steel wool sh-2#"}, {"probe_id": "probe_0561", "target_id": "sroie_574", "qwen_paraphrase": "locate receipt transaction from 02-05-18"}, {"probe_id": "probe_0562", "target_id": "sroie_575", "qwen_paraphrase": "locate receipt transaction from 05-05-18"}, {"probe_id": "probe_0564", "target_id": "sroie_577", "qwen_paraphrase": "locate receipt transaction from rm7.50 z"}, {"probe_id": "probe_0566", "target_id": "sroie_579", "qwen_paraphrase": "locate receipt transaction from rm 11.23"}, {"probe_id": "probe_0567", "target_id": "sroie_580", "qwen_paraphrase": "locate receipt transaction from cylindrical lock 3871ss"}, {"probe_id": "probe_0568", "target_id": "sroie_581", "qwen_paraphrase": "locate receipt transaction from fill in enterprise"}, {"probe_id": "probe_0569", "target_id": "sroie_582", "qwen_paraphrase": "locate receipt transaction from : cr 1805/0201"}, {"probe_id": "probe_0570", "target_id": "sroie_583", "qwen_paraphrase": "locate receipt transaction from 30/05/18 11:01"}, {"probe_id": "probe_0571", "target_id": "sroie_584", "qwen_paraphrase": "locate receipt transaction from 28/05/18 11:41"}, {"probe_id": "probe_0573", "target_id": "sroie_586", "qwen_paraphrase": "locate receipt transaction from : cr 1806/0729"}, {"probe_id": "probe_0577", "target_id": "sroie_590", "qwen_paraphrase": "locate receipt transaction from : sp3-01-147902"}, {"probe_id": "probe_0580", "target_id": "sroie_593", "qwen_paraphrase": "locate receipt transaction from 11/5/2018 13:14:32"}, {"probe_id": "probe_0581", "target_id": "sroie_594", "qwen_paraphrase": "locate receipt transaction from 2249 carlsberg smooth draug"}, {"probe_id": "probe_0582", "target_id": "sroie_595", "qwen_paraphrase": "locate receipt transaction from : 28-04-2018 20:10:51"}, {"probe_id": "probe_0583", "target_id": "sroie_596", "qwen_paraphrase": "locate receipt transaction from : 05-06-2018 20:23:14"}, {"probe_id": "probe_0585", "target_id": "sroie_598", "qwen_paraphrase": "locate receipt transaction from : 28-05-2018 23:28:57"}, {"probe_id": "probe_0586", "target_id": "sroie_599", "qwen_paraphrase": "locate receipt transaction from : 19-05-2018 12:46:58"}, {"probe_id": "probe_0588", "target_id": "sroie_601", "qwen_paraphrase": "locate receipt transaction from 28-11-18"}, {"probe_id": "probe_0591", "target_id": "sroie_604", "qwen_paraphrase": "locate receipt transaction from time: 11:22:00"}, {"probe_id": "probe_0595", "target_id": "sroie_608", "qwen_paraphrase": "locate receipt transaction from date: 23/05/2018"}, {"probe_id": "probe_0597", "target_id": "sroie_610", "qwen_paraphrase": "locate receipt transaction from tel: 04-227 2222"}, {"probe_id": "probe_0600", "target_id": "sroie_613", "qwen_paraphrase": "locate receipt transaction from 03/04/18 19:19"}, {"probe_id": "probe_0601", "target_id": "sroie_614", "qwen_paraphrase": "locate receipt transaction from ocean lc packaging enterprise"}, {"probe_id": "probe_0602", "target_id": "sroie_615", "qwen_paraphrase": "locate receipt transaction from (pusat perniagaan bunga raya"}, {"probe_id": "probe_0605", "target_id": "sroie_618", "qwen_paraphrase": "locate receipt transaction from : 13-06-2018 22:53:19"}, {"probe_id": "probe_0606", "target_id": "sroie_619", "qwen_paraphrase": "locate receipt transaction from : 30-05-2018 13:45:46"}, {"probe_id": "probe_0607", "target_id": "sroie_620", "qwen_paraphrase": "locate receipt transaction from : 17-06-2018 14:07:29"}, {"probe_id": "probe_0609", "target_id": "sroie_622", "qwen_paraphrase": "locate receipt transaction from : 20-06-2018 21:36:11"}, {"probe_id": "probe_0610", "target_id": "sroie_623", "qwen_paraphrase": "locate receipt transaction from : 23-06-2018 20:52:16"}]

human_file = find_file('data/human_queries_200.csv')
if human_file.exists():
    h_df = pd.read_csv(human_file)
    human_raw = [
        {'query_id': f'human_{i:04d}', 'target_id': row['receipt_id'], 'query_text': row['handwritten_paraphrase']}
        for i, row in h_df.iloc[:200].iterrows()
    ]
else:
    human_raw = [
        {'query_id': f'human_{i:04d}', 'target_id': r['receipt_id'], 'query_text': r['handwritten_paraphrase']}
        for i, r in enumerate(human_embedded)
    ]

human_probes = human_raw[:200]

corpus_id_set = set([r['id'] for r in test_corpus])
missing_targets = [q['target_id'] for q in human_probes if q['target_id'] not in corpus_id_set]
print(f'Human Queries Loaded:  {len(human_probes)} (Missing targets in corpus: {len(missing_targets)})')

probes_file = find_file('ai_engine/rag/data/v2/frozen_probes_with_paraphrases.json')
with open(probes_file, 'r', encoding='utf-8') as f:
    p_data = json.load(f)
primary_probes = [{'query_id': p.get('probe_id', f'prim_{i:04d}'), 'target_id': p['target_id'], 'query_text': p['light_paraphrase']} for i, p in enumerate(p_data['test_probes']) if p.get('light_paraphrase')]

qwen_file = find_file('results/v5/qwen_paraphrases_v5.json')
if qwen_file.exists():
    with open(qwen_file, 'r', encoding='utf-8') as f:
        q_raw = json.load(f)
else:
    q_raw = qwen_embedded
qwen_probes = [{'query_id': it.get('probe_id', f'qwen_{i:04d}'), 'target_id': it['target_id'], 'query_text': it['qwen_paraphrase']} for i, it in enumerate(q_raw)]

print(f'Primary Queries:       {len(primary_probes)} (Expected: 306)')
print(f'Secondary Qwen Queries: {len(qwen_probes)} (Expected: 306)')



In [ ]:
# CELL 5: Load ColQwen2-v1.0 & Audit Preprocessing (Validation 4)
import torch
from colpali_engine.models import ColQwen2, ColQwen2Processor

MODEL_ID = "vidore/colqwen2-v1.0"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

print(f"Loading {MODEL_ID} on {device} ({dtype})...", flush=True)
processor = ColQwen2Processor.from_pretrained(MODEL_ID)
model = ColQwen2.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map=device).eval()
print("Model loaded successfully!", flush=True)

# Record processor config
prep_config = {
    "model_id": MODEL_ID,
    "dtype": str(dtype),
    "device": device,
    "image_processor_type": type(processor.image_processor).__name__,
    "tokenizer_vocab_size": processor.tokenizer.vocab_size
}
print("Processor configuration:", json.dumps(prep_config, indent=2))


In [ ]:
# CELL 6: Encode All 813 Document Images on GPU (Live Verification)
from PIL import Image
import time

img_paths = [find_file(r["image_path"]) for r in test_corpus]
corpus_ids = [r["id"] for r in test_corpus]
id_to_idx = {cid: i for i, cid in enumerate(corpus_ids)}

print(f"Encoding {len(img_paths)} document images on GPU...", flush=True)
t0 = time.time()
doc_embeddings = []
batch_size = 4

for b in range(0, len(img_paths), batch_size):
    b_paths = img_paths[b : b + batch_size]
    b_imgs = [Image.open(p).convert("RGB") if p.exists() else Image.new("RGB", (224, 224), (255, 255, 255)) for p in b_paths]
    b_proc = processor.process_images(b_imgs).to(device)
    with torch.no_grad():
        out = model(**b_proc)
        for i in range(len(b_imgs)):
            doc_embeddings.append(out[i].cpu())
    if (b // batch_size) % 25 == 0:
        print(f"  Encoded {min(b + batch_size, len(img_paths))}/{len(img_paths)} images...", flush=True)

encoding_elapsed = time.time() - t0
print(f"Completed live image encoding in {encoding_elapsed:.1f}s!", flush=True)

# Save cached doc embeddings
torch.save(doc_embeddings, working_dir / "colqwen_doc_embeddings.pt")
print("Saved doc embeddings to /kaggle/working/colqwen_doc_embeddings.pt")


In [ ]:
# CELL 7: Fresh Live Scoring & Sub-Sample Stability Sweep (Validations 1 & 2)
import numpy as np

def late_interaction_maxsim(query_emb: torch.Tensor, doc_emb: torch.Tensor) -> float:
    sim = torch.matmul(query_emb.float(), doc_emb.float().t())
    max_sim = torch.max(sim, dim=1).values
    return float(torch.sum(max_sim).item())

def score_query_detailed(q_text: str, tgt_id: str):
    b_q = processor.process_queries([q_text]).to(device)
    with torch.no_grad():
        q_emb = model(**b_q)[0].cpu()
    scores = np.array([late_interaction_maxsim(q_emb, d) for d in doc_embeddings])
    tgt_idx = id_to_idx[tgt_id]
    ranks = np.argsort(-scores)
    pos = np.where(ranks == tgt_idx)[0]
    if len(pos) == 0:
        return 0.0, 999999, 0, 0, 0
    rank = int(pos[0] + 1)
    rr = 1.0 / rank
    return rr, rank, (1 if rank == 1 else 0), (1 if rank <= 5 else 0), (1 if rank <= 10 else 0)

# 1. Score all Human Probes (N=200)
print("Scoring all 200 Human Probes live on GPU...", flush=True)
human_per_query = []
for idx, q in enumerate(human_probes):
    rr, r, h1, h5, h10 = score_query_detailed(q["query_text"], q["target_id"])
    human_per_query.append({
        "query_idx": idx,
        "query_id": q["query_id"],
        "target_id": q["target_id"],
        "query_text": q["query_text"],
        "rank": r,
        "reciprocal_rank": rr,
        "hit1": h1, "hit5": h5, "hit10": h10
    })

# 2. Score Primary (N=306) and Secondary (N=306)
print("Scoring Primary Probes (N=306)...", flush=True)
prim_rrs = [score_query_detailed(q["query_text"], q["target_id"])[0] for q in primary_probes]

print("Scoring Secondary Qwen Probes (N=306)...", flush=True)
qwen_rrs = [score_query_detailed(q["query_text"], q["target_id"])[0] for q in qwen_probes]

# Partition Human Analysis
human_rrs_200 = np.array([d["reciprocal_rank"] for d in human_per_query])
human_rrs_100_tier1 = human_rrs_200[:100]
human_rrs_100_tier2 = human_rrs_200[100:200]

# Sub-sample sweep across 5 seeds
stability_runs = [
    {"Cohort": "Human All (N=200)", "N": 200, "MRR": float(np.mean(human_rrs_200)), "Hit1": float(np.mean([d['hit1'] for d in human_per_query])), "Hit5": float(np.mean([d['hit5'] for d in human_per_query])), "Hit10": float(np.mean([d['hit10'] for d in human_per_query]))},
    {"Cohort": "Human Tier 1 (First 100)", "N": 100, "MRR": float(np.mean(human_rrs_100_tier1)), "Hit1": float(np.mean([d['hit1'] for d in human_per_query[:100]])), "Hit5": float(np.mean([d['hit5'] for d in human_per_query[:100]])), "Hit10": float(np.mean([d['hit10'] for d in human_per_query[:100]]))},
    {"Cohort": "Human Tier 2 (Second 100)", "N": 100, "MRR": float(np.mean(human_rrs_100_tier2)), "Hit1": float(np.mean([d['hit1'] for d in human_per_query[100:200]])), "Hit5": float(np.mean([d['hit5'] for d in human_per_query[100:200]])), "Hit10": float(np.mean([d['hit10'] for d in human_per_query[100:200]]))},
]

for s in [42, 123, 456, 789, 1024]:
    rng = np.random.default_rng(s)
    idx_sample = rng.choice(200, size=100, replace=False)
    sample_rrs = human_rrs_200[idx_sample]
    stability_runs.append({
        "Cohort": f"Human Random Sample N=100 (Seed {s})",
        "N": 100,
        "MRR": float(np.mean(sample_rrs)),
        "Hit1": float(np.mean([human_per_query[i]['hit1'] for i in idx_sample])),
        "Hit5": float(np.mean([human_per_query[i]['hit5'] for i in idx_sample])),
        "Hit10": float(np.mean([human_per_query[i]['hit10'] for i in idx_sample]))
    })

df_stability = pd.DataFrame(stability_runs)
print("\n=== STABILITY & PARTITION SWEEP (Validation 1 & 2) ===")
print(df_stability.to_markdown(index=False))


In [ ]:
# CELL 8: 10,000 Bootstrap Uncertainty Estimates (Validation 5)
def bootstrap_ci(arr, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    boot = [float(np.mean(rng.choice(arr, size=len(arr), replace=True))) for _ in range(n_boot)]
    se = float(np.std(boot))
    ci_lower = float(np.percentile(boot, 2.5))
    ci_upper = float(np.percentile(boot, 97.5))
    return se, ci_lower, ci_upper

ci_results = []
suites_eval = [
    ("Primary Gen. (gpt-4o-mini, N=306)", np.array(prim_rrs)),
    ("Secondary Gen. (Qwen2.5-7B, N=306)", np.array(qwen_rrs)),
    ("Human-Authored Queries (N=100)", human_rrs_100_tier1),
    ("Human-Authored Queries (N=200)", human_rrs_200),
    ("Human Tier 2 Queries (N=100)", human_rrs_100_tier2),
]

for name, arr in suites_eval:
    se, l, u = bootstrap_ci(arr, n_boot=10000, seed=42)
    ci_results.append({
        "Suite": name,
        "N": len(arr),
        "Measured_MRR": round(float(np.mean(arr)), 4),
        "Std_Error": round(se, 4),
        "95%_CI_Lower": round(l, 4),
        "95%_CI_Upper": round(u, 4),
        "95%_CI_Formatted": f"[{l:.4f}, {u:.4f}]"
    })

df_ci = pd.DataFrame(ci_results)
print("\n=== 10,000 BOOTSTRAP UNCERTAINTY ESTIMATES (Validation 5) ===")
print(df_ci.to_markdown(index=False))


In [ ]:
# CELL 9: Cross-System Parity & Alignment (Validation 3)
# Baseline comparisons for Human N=100 and N=200
cross_system = [
    {"System": "Word BM25", "Human_N100_MRR": 0.6353, "Human_N200_MRR": 0.5626, "Source": "Unified Test Index (N=813)"},
    {"System": "Char 3-5g BM25", "Human_N100_MRR": 0.6059, "Human_N200_MRR": 0.5605, "Source": "Unified Test Index (N=813)"},
    {"System": "Dense BGE-M3", "Human_N100_MRR": 0.5020, "Human_N200_MRR": 0.4909, "Source": "Unified Test Index (N=813)"},
    {"System": "ColQwen2-v1.0 (Live GPU)", "Human_N100_MRR": round(float(np.mean(human_rrs_100_tier1)), 4), "Human_N200_MRR": round(float(np.mean(human_rrs_200)), 4), "Source": "Live Tesla T4 GPU Execution"},
]

df_cross = pd.DataFrame(cross_system)
print("\n=== CROSS-SYSTEM COMPARISON TABLE (Table XI Authoritative Basis) ===")
print(df_cross.to_markdown(index=False))


In [ ]:
# CELL 10: Master Artifact Packaging & Summary Export
import zipfile

out_dir = working_dir / "results" / "validation_run"
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Save stability summary
df_stability.to_csv(out_dir / "human_subsample_stability.csv", index=False)

# 2. Save per-query rankings
df_per_query = pd.DataFrame(human_per_query)
df_per_query.to_csv(out_dir / "human_colqwen_per_query_n200.csv", index=False)

# 3. Save CI validation
df_ci.to_csv(out_dir / "bootstrap_ci_validation.csv", index=False)

# 4. Save cross-system comparison
df_cross.to_csv(out_dir / "cross_system_alignment.csv", index=False)

# 5. Save audit metadata
audit_report = {
    "corpus_id_sha256": corpus_hash,
    "corpus_size": len(test_corpus),
    "human_probes_count": len(human_probes),
    "torch_version": torch.__version__,
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "encoding_time_seconds": round(encoding_elapsed, 1),
    "mrr_human_100": round(float(np.mean(human_rrs_100_tier1)), 4),
    "mrr_human_200": round(float(np.mean(human_rrs_200)), 4),
}
with open(out_dir / "audit_report.json", "w") as f:
    json.dump(audit_report, f, indent=2)

# Create ZIP archive
zip_path = working_dir / "colqwen2_human_validation_results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in out_dir.glob("*"):
        zf.write(f, arcname=f.name)

print(f"\nAll validation artifacts successfully packaged to: {zip_path}")
print(f"Archive size: {zip_path.stat().st_size / 1024:.1f} KB")
